# Broadcast Message Bus (publish / subscribe)

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Concurrency, Hash Tables · **Difficulty/Frequency:** Common (5/10)

> **Language note.** The official answer is Java (`ConcurrentHashMap` + `CopyOnWriteArrayList`). This notebook builds the *same design* in Python: a `threading.Lock`-guarded topic map plus an **explicit tuple snapshot** where Java gets the snapshot for free from its collection type. The Java reference is preserved verbatim in [`README.md`](README.md).

## Concepts

**What this problem is really testing:**
- **Snapshot iteration** — how to safely walk a collection that someone else may be mutating
- Designing an **opaque handle** (the subscription token) that identifies exactly one thing
- Knowing that the interesting part of pub/sub is not delivery, it is the **races around delivery**

**First-principles primer — what is each piece?**

- **Publish/subscribe.** Publishers do not know who is listening; subscribers do not know who is sending. The bus in the middle is the only thing that knows both. That decoupling is the point — it is why MongoDB change streams, Kafka topics and DOM event listeners all look like this.
- **Topic.** A named channel. `topic -> list of handlers` is the whole data model; everything else is bookkeeping.
- **Handler / callback.** A function the bus calls when a message arrives. Crucially, **the bus does not control what that function does** — and it may well call back into the bus.
- **Snapshot iteration.** Taking a frozen copy of a collection and walking *that*, so concurrent changes to the original cannot disturb the walk. Java's `CopyOnWriteArrayList` builds this in: every write replaces the whole backing array, so a reader that grabbed the old array keeps a consistent view.
- **Token.** An opaque string handed back by `subscribe` and passed to `unsubscribe`. The caller cannot inspect it; it just holds it.

**The one thing this problem is actually about:**

> *"Unsubscribing during a publish must not cause exceptions or missed deliveries to other subscribers."*

That single requirement is the whole question. Consider the obvious implementation:

```python
for handler in self.topics[topic]:      # iterating the LIVE list
    handler(message)                    # ...which this call may mutate
```

If a handler calls `unsubscribe`, the list shrinks **while the for-loop is walking it**. In Java that raises `ConcurrentModificationException`. In Python it is worse — no exception, but the loop's internal index now points past a removed element, so **the next subscriber is silently skipped**. A missed delivery with no error is the hardest kind of bug to find.

**The fix, in one line:** iterate a **snapshot**, not the live list.

```python
for handler in tuple(self.topics[topic]):   # a frozen copy
```

Mutations during the loop land on the real list and take effect on the *next* publish. The current one runs to completion over the membership that existed when it started — which is exactly the contract you want, and the contract you can explain.

**Simple worked example.** Topic `orders` has handlers A, B, C. A publish begins; handler A unsubscribes B.

| | live list | snapshot being walked | delivered |
|---|---|---|---|
| start | `[A, B, C]` | `(A, B, C)` | — |
| A runs, removes B | `[A, C]` | `(A, B, C)` | A |
| next | `[A, C]` | `(A, B, C)` | A, **B** |
| next | `[A, C]` | `(A, B, C)` | A, B, C |

Everyone present at the start still got the message; B's removal takes effect from the next publish onward. **Without** the snapshot, C would have been silently skipped.

## Problem Statement

| Method | Behaviour |
|---|---|
| `subscribe(topic, handler) -> token` | Register `handler`; return an opaque token |
| `publish(topic, message)` | Call every current subscriber of `topic` |
| `unsubscribe(token)` | Remove that one subscription |

**Requirements**

1. Multiple subscribers on a topic each receive **every** message.
2. **Unsubscribing during a publish must not raise, and must not cause missed deliveries to others.**
3. Thread-safe.
4. Publishing to a topic with no subscribers is a **no-op**, not an error.

**Follow-up:** durable subscriptions — a subscriber reconnects and receives what it missed. What data structure bounds the memory?

### Approach 1 — Naive (iterate the live list under one big lock)

**Idea:** a plain dict of topic to list, with a lock around every method.

It looks thread-safe, and it fails on **two** counts:

1. **`publish` iterates the live list.** A handler that unsubscribes mutates the list mid-loop, and Python silently skips the following subscriber — no exception, just a lost message.
2. **The handler is called while holding the lock.** A handler that calls `subscribe` or `publish` re-enters and **deadlocks instantly** on a non-reentrant lock. Handlers are arbitrary user code; you cannot assume they will not call back in.

**Time complexity:** O(n) per publish, but every publish across *all* topics is serialised.

**Space complexity:** O(s) for s subscriptions.

In [ ]:
import threading
import uuid
from typing import Any, Callable, Dict, List, Optional, Tuple

Handler = Callable[[str], None]


class NaiveMessageBus:
    """Baseline: looks thread-safe, drops messages, and deadlocks on re-entrant handlers."""

    def __init__(self) -> None:
        self.topics: Dict[str, List[Tuple[str, Handler]]] = {}
        self.lock = threading.Lock()

    def subscribe(self, topic: str, handler: Handler) -> str:
        token = uuid.uuid4().hex
        with self.lock:
            self.topics.setdefault(topic, []).append((token, handler))
        return token

    def publish(self, topic: str, message: str) -> None:
        with self.lock:                       # BUG 2: the lock is held across user code
            for _, handler in self.topics.get(topic, []):   # BUG 1: iterating the LIVE list
                handler(message)

    def unsubscribe(self, token: str) -> None:
        with self.lock:
            for topic, subs in list(self.topics.items()):
                for i, (t, _) in enumerate(subs):
                    if t == token:
                        subs.pop(i)           # mutates the list a publish may be walking
                        return


class NaiveUnlockedBus(NaiveMessageBus):
    """Same, minus the lock during publish - so the mid-iteration skip is visible on its own."""

    def publish(self, topic: str, message: str) -> None:
        for _, handler in self.topics.get(topic, []):       # still the LIVE list
            handler(message)

### Approach 2 — Optimal (snapshot the subscriber tuple, call handlers outside the lock)

**Idea:** three changes, each fixing a specific failure.

1. **`publish` takes a snapshot.** `self.topics.get(topic, ())` returns a **tuple**, and every mutation *replaces* it rather than editing in place. A publish that grabbed the old tuple keeps walking a stable, complete list. This is precisely what Java's `CopyOnWriteArrayList` does — copy on write, so reads never need a lock and never see a torn view.
2. **Handlers run with no lock held.** The lock covers only the dictionary read. Since handlers are arbitrary user code that may call `subscribe`, `publish` or `unsubscribe`, holding a lock across them invites deadlock and serialises the entire bus behind the slowest listener.
3. **A `token -> (topic, handler)` index.** `unsubscribe` becomes O(1) on the lookup instead of scanning every topic. It also makes the token genuinely opaque: it identifies *one subscription*, so two subscriptions of the same handler to the same topic remain independently removable.

**Why copy-on-write is the right trade here.** A pub/sub bus is overwhelmingly read-heavy — many publishes, rare subscription changes. Copy-on-write puts *all* the cost on the rare operation (O(n) to rebuild the tuple) and *none* on the common one (a lock-free tuple read). If that ratio inverted — thousands of subscribers with constant churn — a `ReadWriteLock` around a mutable list would be the better fit. Naming that boundary is what shows you understand the choice rather than reciting it.

**Time complexity:** O(n) per `publish` over that topic's n subscribers; O(n) per `subscribe`/`unsubscribe` (the copy); O(1) token lookup.

**Space complexity:** O(s) total subscriptions.

In [ ]:
class MessageBus:
    """Copy-on-write topic map. Handlers are invoked outside the lock, over a snapshot."""

    def __init__(self) -> None:
        self._topics: Dict[str, Tuple[Tuple[str, Handler], ...]] = {}   # topic -> tuple of (token, handler)
        self._index: Dict[str, str] = {}                                # token -> topic  (O(1) unsubscribe)
        self._lock = threading.Lock()

    def subscribe(self, topic: str, handler: Handler) -> str:
        token = uuid.uuid4().hex
        with self._lock:
            current = self._topics.get(topic, ())
            self._topics[topic] = current + ((token, handler),)   # REPLACE the tuple, never mutate
            self._index[token] = topic
        return token

    def publish(self, topic: str, message: str) -> int:
        with self._lock:
            subs = self._topics.get(topic, ())    # one atomic read; () => publishing is a no-op
        # Lock released. Handlers may now subscribe/unsubscribe/publish without deadlocking,
        # and `subs` is a frozen snapshot so those changes cannot disturb this loop.
        for _, handler in subs:
            handler(message)
        return len(subs)

    def unsubscribe(self, token: str) -> bool:
        with self._lock:
            topic = self._index.pop(token, None)
            if topic is None:
                return False                      # unknown or already-removed token: not an error
            remaining = tuple(s for s in self._topics.get(topic, ()) if s[0] != token)
            if remaining:
                self._topics[topic] = remaining
            else:
                del self._topics[topic]           # never leave an empty topic behind
            return True

    def subscriber_count(self, topic: str) -> int:
        with self._lock:
            return len(self._topics.get(topic, ()))

    def topic_count(self) -> int:
        with self._lock:
            return len(self._topics)

### Approach 3 — Isolating a failing handler

**Idea:** a handler is user code, and user code raises. In Approach 2 an exception from the *first* handler propagates out of `publish` — so subscribers 2..n never receive the message, and the publisher gets an error about somebody else's bug.

Almost always the right policy for a broadcast bus is **catch, record, continue**: one broken subscriber must not censor the others. But the policy has to be a deliberate choice, stated aloud:

| Policy | When it is right |
|---|---|
| catch & continue (here) | broadcast/notification — delivery to others matters more |
| propagate | the publisher genuinely depends on every handler succeeding |
| catch, and unsubscribe the offender | a handler that keeps failing is poisoning the topic |

Errors are collected and returned rather than swallowed, so the caller can log or alert. Silently discarding them is the one option that is always wrong.

**Time complexity:** unchanged.

**Space complexity:** unchanged, plus the errors collected in one call.

In [ ]:
class ResilientMessageBus(MessageBus):
    """One failing handler must not stop delivery to the others."""

    def publish(self, topic: str, message: str) -> List[Tuple[str, BaseException]]:
        with self._lock:
            subs = self._topics.get(topic, ())
        errors: List[Tuple[str, BaseException]] = []
        for token, handler in subs:
            try:
                handler(message)
            except Exception as exc:              # not BaseException: let KeyboardInterrupt through
                errors.append((token, exc))       # record it, keep delivering
        return errors

### Follow-up — durable subscriptions, with bounded memory

**Idea:** the bus above is **fire-and-forget** — a subscriber that is offline when a message is published misses it permanently. A *durable* subscription survives disconnection.

Two ideas make it work:

- **Keep a per-topic log** of published messages, each with a monotonically increasing **sequence number**, and remember a **cursor** per durable subscription. Reconnecting means "replay everything after my cursor". This is exactly Kafka's consumer-offset model, and MongoDB change streams' resume token.
- **Bound the memory with a ring buffer.** Retaining every message forever is not an option, so keep only the last `capacity` per topic — `collections.deque(maxlen=capacity)` is a ring buffer in one line. Old messages are evicted automatically.

**The consequence you must name:** a subscriber offline long enough for its cursor to fall behind the retention window **cannot be caught up** — the messages are simply gone. That is not a bug, it is the price of bounded memory, and every real system has it (Kafka's `OFFSET_OUT_OF_RANGE`, MongoDB's "resume token no longer in the oplog"). Detecting it and telling the subscriber honestly is what a good implementation does; silently resuming from an arbitrary point is what a bad one does.

**Time complexity:** O(1) to append; O(m) to replay m missed messages.

**Space complexity:** **O(topics × capacity)** — bounded by construction, independent of uptime.

In [ ]:
from collections import deque


class DurableMessageBus(MessageBus):
    """Per-topic ring buffer + per-subscription cursor => replay what was missed, bounded memory."""

    def __init__(self, capacity: int = 100) -> None:
        super().__init__()
        self.capacity = capacity
        self._log: Dict[str, Any] = {}          # topic -> deque of (seq, message), maxlen=capacity
        self._seq: Dict[str, int] = {}          # topic -> next sequence number
        self._cursor: Dict[str, int] = {}       # token -> last sequence number delivered

    def publish(self, topic: str, message: str) -> int:
        with self._lock:
            seq = self._seq.get(topic, 0)
            self._seq[topic] = seq + 1
            log = self._log.setdefault(topic, deque(maxlen=self.capacity))
            log.append((seq, message))          # maxlen makes eviction automatic - a ring buffer
            subs = self._topics.get(topic, ())
            for token, _ in subs:
                self._cursor[token] = seq       # online subscribers are up to date by definition
        for _, handler in subs:
            handler(message)
        return len(subs)

    def resume(self, token: str, topic: str, handler: Handler) -> Tuple[int, bool]:
        """Reconnect. Returns (messages replayed, whether a gap was detected)."""
        with self._lock:
            last_seen = self._cursor.get(token, -1)
            log = list(self._log.get(topic, ()))
            self._topics[topic] = self._topics.get(topic, ()) + ((token, handler),)
            self._index[token] = topic
            if log:
                self._cursor[token] = log[-1][0]
        missed = [(s, m) for s, m in log if s > last_seen]
        # A gap: the oldest retained message is already newer than where we left off.
        gap = bool(log) and log[0][0] > last_seen + 1
        for _, m in missed:
            handler(m)
        return len(missed), gap

    def disconnect(self, token: str) -> None:
        """Go offline but KEEP the cursor, so resume() can catch up later."""
        with self._lock:
            topic = self._index.pop(token, None)
            if topic is None:
                return
            remaining = tuple(s for s in self._topics.get(topic, ()) if s[0] != token)
            if remaining:
                self._topics[topic] = remaining
            else:
                del self._topics[topic]
            # NOTE: self._cursor[token] is deliberately left in place.

## Verification

The requirements here are all about *races*, so they have to be run, not argued. These checks cover the four stated requirements, the unsubscribe-during-publish case that is the heart of the question, handler failures, and real concurrent threads.

In [ ]:
import random
from concurrent.futures import ThreadPoolExecutor

# --- Requirement 1: every subscriber on a topic receives every message ---
bus = MessageBus()
a, b = [], []
bus.subscribe("orders", a.append)
bus.subscribe("orders", b.append)
bus.publish("orders", "m1")
bus.publish("orders", "m2")
assert a == ["m1", "m2"] and b == ["m1", "m2"]

# --- Requirement 4: publishing to a topic with no subscribers is a no-op ---
assert bus.publish("nobody-here", "m") == 0
assert bus.topic_count() == 1, "a publish must not create a topic"

# --- Topics are isolated from one another ---
other = []
bus.subscribe("shipments", other.append)
bus.publish("orders", "m3")
assert other == [], "an orders message must not reach shipments"

# --- unsubscribe removes exactly one subscription ---
received = []
tok = bus.subscribe("t", received.append)
bus.publish("t", "before")
assert bus.unsubscribe(tok) is True
bus.publish("t", "after")
assert received == ["before"]
assert bus.unsubscribe(tok) is False, "unsubscribing twice is a no-op, not an error"
assert bus.unsubscribe("garbage-token") is False

# Two subscriptions of the SAME handler must be independently removable
shared = []
t1 = bus.subscribe("dup", shared.append)
t2 = bus.subscribe("dup", shared.append)
bus.publish("dup", "x")
assert shared == ["x", "x"], "the same handler subscribed twice receives twice"
bus.unsubscribe(t1)
bus.publish("dup", "y")
assert shared == ["x", "x", "y"], "removing one subscription must leave the other"
bus.unsubscribe(t2)

# Emptied topics are cleaned up, not left behind
assert "dup" not in bus._topics and "t" not in bus._topics

# --- REQUIREMENT 2: unsubscribing DURING a publish ---
# The middle handler removes the third one. Everyone present at the start must still be called.
bus = MessageBus()
order = []
tokens = {}


def h1(msg):
    order.append("h1")


def h2(msg):
    order.append("h2")
    bus.unsubscribe(tokens["h3"])       # remove a LATER subscriber, mid-publish


def h3(msg):
    order.append("h3")


tokens["h1"] = bus.subscribe("race", h1)
tokens["h2"] = bus.subscribe("race", h2)
tokens["h3"] = bus.subscribe("race", h3)

bus.publish("race", "m")
assert order == ["h1", "h2", "h3"], f"snapshot must deliver to everyone present at start: {order}"
order.clear()
bus.publish("race", "m")
assert order == ["h1", "h2"], "the removal takes effect from the NEXT publish"

# A handler removing ITSELF mid-publish
bus = MessageBus()
seen = []
self_tok = {}


def suicidal(msg):
    seen.append(msg)
    bus.unsubscribe(self_tok["t"])


self_tok["t"] = bus.subscribe("s", suicidal)
bus.subscribe("s", seen.append)
bus.publish("s", "one")
assert seen == ["one", "one"], "the second subscriber must still be reached"
bus.publish("s", "two")
assert seen == ["one", "one", "two"], "only the surviving subscriber remains"

# A handler that SUBSCRIBES mid-publish must not receive the in-flight message
bus = MessageBus()
late = []
def adder(msg):
    bus.subscribe("g", late.append)
bus.subscribe("g", adder)
bus.publish("g", "first")
assert late == [], "a subscriber added mid-publish must not receive that same message"
bus.publish("g", "second")
# The snapshot was (adder, late.append). adder ran and added a THIRD subscriber, which -
# correctly - does not receive "second" either. So exactly one append happened.
assert late == ["second"], f"only the subscriber present at publish time is called: {late}"
assert bus.subscriber_count("g") == 3

# A handler that PUBLISHES re-enters the bus - this deadlocks if the lock spans handler calls
bus = MessageBus()
chain = []
bus.subscribe("b", chain.append)
bus.subscribe("a", lambda m: bus.publish("b", m + "!"))
done = threading.Event()
threading.Thread(target=lambda: (bus.publish("a", "hi"), done.set()), daemon=True).start()
assert done.wait(timeout=3.0), "a re-entrant handler deadlocked: the lock spans user code"
assert chain == ["hi!"]

# --- The naive bus demonstrates the bug this design exists to prevent ---
naive = NaiveUnlockedBus()
hit = []
ntok = {}
naive.subscribe("x", lambda m: hit.append("first"))
ntok["second"] = naive.subscribe("x", lambda m: (hit.append("second"),
                                                 naive.unsubscribe(ntok["second"])))
naive.subscribe("x", lambda m: hit.append("third"))
naive.publish("x", "m")
assert hit == ["first", "second"], (
    f"the naive bus silently SKIPS the third subscriber: {hit}"
)

# --- Requirement 3 + failing handlers: real threads ---
bus = ResilientMessageBus()
bad_token = bus.subscribe("mixed", lambda m: (_ for _ in ()).throw(ValueError("boom")))
good = []
bus.subscribe("mixed", good.append)
errors = bus.publish("mixed", "payload")
assert good == ["payload"], "a raising handler must not censor the ones after it"
assert len(errors) == 1 and isinstance(errors[0][1], ValueError)
assert errors[0][0] == bad_token, "the caller learns WHICH subscription failed"

# Concurrent publishers, subscribers and unsubscribers over the same bus
bus = MessageBus()
counts = {}
counts_lock = threading.Lock()


def make_handler(name):
    def h(msg):
        with counts_lock:
            counts[name] = counts.get(name, 0) + 1
    return h


stable = [bus.subscribe("hot", make_handler(f"s{i}")) for i in range(5)]
rng = random.Random(3)


def churn(worker):
    for i in range(40):
        act = rng.random()
        if act < 0.5:
            bus.publish("hot", f"m{worker}-{i}")
        elif act < 0.8:
            t = bus.subscribe("hot", make_handler(f"tmp{worker}-{i}"))
            bus.unsubscribe(t)
        else:
            bus.subscriber_count("hot")


with ThreadPoolExecutor(max_workers=8) as ex:
    list(ex.map(churn, range(8)))       # no exception escaping == thread-safe enough to pass

assert bus.subscriber_count("hot") == 5, "every temporary subscription must be cleaned up"
for t in stable:
    assert bus.unsubscribe(t)
assert bus.subscriber_count("hot") == 0

# --- Durable subscriptions ---
d = DurableMessageBus(capacity=5)
inbox = []
tok = d.subscribe("news", inbox.append)
d.publish("news", "n0")
d.publish("news", "n1")
assert inbox == ["n0", "n1"]

d.disconnect(tok)                       # go offline, KEEP the cursor
d.publish("news", "n2")
d.publish("news", "n3")
assert inbox == ["n0", "n1"], "an offline subscriber receives nothing live"

replayed, gap = d.resume(tok, "news", inbox.append)
assert (replayed, gap) == (2, False), (replayed, gap)
assert inbox == ["n0", "n1", "n2", "n3"], "exactly the missed messages are replayed"

d.publish("news", "n4")
assert inbox[-1] == "n4", "after resuming, live delivery continues"

# Falling behind the retention window is DETECTED, not silently papered over
d.disconnect(tok)
for i in range(20):                     # capacity is 5, so the cursor falls off the end
    d.publish("news", f"flood{i}")
replayed, gap = d.resume(tok, "news", inbox.append)
assert gap is True, "a subscriber past the retention window must be told it lost messages"
assert replayed == 5, "only what the ring buffer still holds can be replayed"
assert len(d._log["news"]) == 5, "the ring buffer is bounded regardless of uptime"

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **Durable subscriptions and bounding memory.** Built above. The two moving parts are a **per-topic log with sequence numbers** and a **per-subscription cursor**; the bound comes from a **ring buffer** (`deque(maxlen=...)`). The honest consequence — a subscriber can fall off the end and *cannot* be caught up — is the part worth volunteering, because it is what every real system (Kafka, MongoDB's oplog) actually does. Detect the gap and report it; do not silently resume from wherever.
- **A handler that throws.** Covered by `ResilientMessageBus`. State the policy explicitly rather than defaulting into one: catch-and-continue for a broadcast bus, propagate if the publisher truly depends on every handler, or catch-and-unsubscribe for a handler that keeps failing (a circuit breaker for listeners). Whatever you choose, return or log the errors — swallowing them is the only universally wrong answer.
- **Wildcard / hierarchical topics** (`orders.*`, `orders.created`). Split each topic on `.` and store subscriptions in a **trie**, with `*` and `#` as special child edges. Publishing walks the trie once, collecting handlers from every node that matches — O(depth × branching) rather than testing the pattern against every topic. This is MQTT's and RabbitMQ's topic-exchange model exactly.
- **When copy-on-write stops paying.** Every subscribe/unsubscribe rebuilds the whole tuple, so a topic with 10,000 subscribers and constant churn does 10,000 copies per change. Switch to a `ReadWriteLock` around a mutable list (many concurrent readers, exclusive writer) — but note you must then **copy the list inside the read lock** before calling handlers, or you are back to iterating live state. The snapshot requirement never goes away; only who pays for it does.
- **Going distributed.** None of the in-process primitives survive: there is no shared memory, so a lock is meaningless across machines. You need serialization, a network protocol, and a broker (or a gossip layer) holding the topic registry. Delivery semantics stop being free and become a *choice* — at-most-once, at-least-once, or exactly-once — each with a different cost. The per-topic log with cursors is exactly what makes at-least-once possible, which is why Kafka is built around it.
- **Synchronous delivery is a design decision.** `publish` here calls handlers on the publisher's own thread, so a slow subscriber slows the publisher. The alternative is to hand each message to a queue drained by worker threads — publishers return immediately, at the cost of losing ordering guarantees and needing backpressure when a subscriber cannot keep up.

## Empirical complexity check

Two things worth measuring, and they pull in opposite directions.

**Publish** is O(n) in the subscribers on that topic — unavoidable, since every one of them must be called. Both implementations pay it.

**Subscribe/unsubscribe** is where copy-on-write charges its rent: rebuilding the tuple is O(n), so a churn-heavy workload is **O(n²)** overall. The naive list's `append` is O(1). That is the trade laid bare — and exactly why you would abandon copy-on-write for a topic with thousands of subscribers and constant churn.

| Growth when the subscriber count doubles | What it means |
|---|---|
| ~2x | linear — the expected cost of delivering to twice as many handlers |
| ~4x | quadratic — n copies of an n-element tuple; the copy-on-write ceiling |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

PUBLISHES = 200


def make_bus(n):
    return (n,)


def run_publish(n):
    bus = MessageBus()
    sink = []
    for _ in range(n):
        bus.subscribe("t", sink.append)
    for i in range(PUBLISHES):
        bus.publish("t", "m")               # O(n) per publish - unavoidable


def run_churn_cow(n):
    bus = MessageBus()
    sink = []
    tokens = [bus.subscribe("t", sink.append) for _ in range(n)]   # each subscribe copies the tuple
    for t in tokens:
        bus.unsubscribe(t)                                          # ...and so does each unsubscribe


def run_churn_naive(n):
    bus = NaiveMessageBus()
    sink = []
    tokens = [bus.subscribe("t", sink.append) for _ in range(n)]   # O(1) append
    for t in tokens:
        bus.unsubscribe(t)                                          # O(total) scan


benchmark(
    {"publish x200 - O(n) per publish": run_publish,
     "subscribe+unsubscribe n - copy-on-write O(n^2)": run_churn_cow,
     "subscribe+unsubscribe n - naive list": run_churn_naive},
    make_bus,
    sizes=[250, 500, 1000, 2000],
    repeats=2,
)

## Patterns learned

- **Never iterate a collection that the code inside the loop can mutate.** Take a snapshot. In Java that is `CopyOnWriteArrayList`; in Python it is `tuple(...)` or `list(...)`. The Python version of this bug is the dangerous one, because it does not raise — it silently skips elements.
- **Never hold a lock while calling code you do not control.** Callbacks, I/O, and anything user-supplied can re-enter, block, or take forever. Read the shared state under the lock, release, *then* act. The same rule that shaped the [Connection Pool](../4.%20Connection_Pool/4.%20Connection_Pool.ipynb) answer.
- **Copy-on-write trades write cost for free reads.** It is the right structure when reads vastly outnumber writes, and the wrong one when they do not. Knowing where the crossover sits is the difference between reciting a data structure and choosing one.
- **A handle should identify one thing, opaquely.** Not the topic (ambiguous with two subscriptions), not the handler (says nothing about which topic) — a token for *this subscription*, plus an index so removal is O(1).
- **Decide what happens when someone else's code fails.** Catch, propagate, or evict — but choose deliberately and surface the error. A broadcast that one bad listener can silence is a broken broadcast.
- **Clean up empty containers.** An emptied topic left in the map is the same leak as an emptied posting list in an index or an emptied bucket in a sparse index. Same invariant discipline, three different problems.
- **A log plus a cursor turns fire-and-forget into replayable.** Sequence numbers and per-consumer offsets are how every durable messaging system works — and a **ring buffer** is what makes the memory bounded, at the cost of an honest "you fell too far behind".
- **Concurrency claims must be executed, not reasoned about.** The mid-publish unsubscribe, the re-entrant handler, the deadlock — all three pass a code review and fail a test.